In [1]:
%pip install pypdf langchain-text-splitters chromadb sentence-transformers ollama rank_bm25

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import warnings
from pathlib import Path
from pypdf import PdfReader

# Silence progress bars / warnings that can hang the notebook UI
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

# Resolve the project root whether this notebook is run from the repo root
# or from notebooks/ directly, so paths work on any machine.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").is_dir() and (PROJECT_ROOT.parent / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
CHROMA_PATH = PROJECT_ROOT / "chroma_db"


def extract_text_from_pdfs(folder_path: Path) -> list[dict]:
    extracted_data = []

    pdf_files = sorted(
        f for f in folder_path.iterdir()
        if f.is_file() and f.suffix.lower() == ".pdf"
    )

    if not pdf_files:
        print(f"No PDF files found in: {folder_path}")
        return extracted_data

    print(f"Found {len(pdf_files)} PDF file(s). Extracting text...\n")

    for pdf_path in pdf_files:
        try:
            reader = PdfReader(pdf_path)
            total_pages = len(reader.pages)

            for page_num, page in enumerate(reader.pages, start=1):
                page_text = page.extract_text() or ""
                cleaned_text = page_text.strip()

                if cleaned_text:
                    extracted_data.append({
                        "file_name": pdf_path.name,
                        "file_path": str(pdf_path),
                        "page_number": page_num,
                        "total_pages": total_pages,
                        "text": cleaned_text
                    })
            print(f"✓ Processed: {pdf_path.name} ({total_pages} pages)")
        except Exception as e:
            print(f"✗ Failed to read {pdf_path.name}: {e}")

    return extracted_data

records = extract_text_from_pdfs(DATA_DIR)
print(f"\nTotal raw pages extracted: {len(records)}")

Found 7 PDF file(s). Extracting text...

✓ Processed: 01_Resume_Writing_Best_Practices.pdf (3 pages)
✓ Processed: 02_How_to_Analyze_a_Job_Description.pdf (3 pages)
✓ Processed: 03_Behavioral_Interview_Framework.pdf (3 pages)
✓ Processed: 04_Technical_Interview_and_System_Design.pdf (3 pages)
✓ Processed: 05_Salary_Negotiation_Playbook.pdf (2 pages)
✓ Processed: 06_Career_Growth_and_Personal_Branding.pdf (2 pages)
✓ Processed: 07_Data_Roles_Roadmaps_and_Tools.pdf (3 pages)

Total raw pages extracted: 19


In [3]:
records[0]

{'file_name': '01_Resume_Writing_Best_Practices.pdf',
 'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
 'page_number': 1,
 'total_pages': 3,
 'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat actually 

In [4]:
import re

def clean_document_text(text: str) -> str:
    # 1. Strip repeated disclaimer/legal boilerplate that trails each page
    disclaimer_pattern = r"This guide reflects widely-observed hiring practices.*?industry, and location\."
    text = re.sub(disclaimer_pattern, "", text, flags=re.DOTALL | re.IGNORECASE)

    # 2. Remove stray control characters left over from PDF extraction (\x7f)
    text = text.replace('\x7f', '')

    # 3. Rejoin words split across a line break by a hyphen (e.g. "re-\nsponsible" -> "responsible")
    text = re.sub(r'(\w+)-\n(\w+)', r'\1\2', text)

    # 4. Normalize all bullet glyphs (■, •, ▪, ►, *) to a single "- " marker
    text = re.sub(r'(?:[\r\n]+\s*)+[■•▪►\*\-]\s*', r'\n- ', text)

    # 5. Clean up orphaned bullet markers left by the normalization above
    text = re.sub(r'\n[-*]\s*\n', '\n- ', text)

    # 6. Insert a paragraph break before numbered section headers (e.g. "1. Intro")
    #    so the chunker below can split at real section boundaries instead of
    #    cutting mid-section.
    text = re.sub(r'\n(\d+\.\s)', r'\n\n\1', text)

    # 7. Collapse extra whitespace and blank lines
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n{3,}', '\n\n', text)

    return text.strip()

def preprocess_records(raw_records: list[dict]) -> list[dict]:
    cleaned = []
    for record in raw_records:
        cleaned_content = clean_document_text(record["text"])
        if len(cleaned_content) > 30:
            cleaned.append({
                "file_name": record["file_name"],
                "file_path": record["file_path"],
                "page_number": record["page_number"],
                "total_pages": record["total_pages"],
                "text": cleaned_content
            })
    print(f"✓ Cleaned {len(cleaned)} pages successfully.")
    return cleaned

cleaned_records = preprocess_records(records)

✓ Cleaned 18 pages successfully.


In [5]:
cleaned_records[:3]

[{'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'file_path': 'C:\\Users\\Ahmed\\Desktop\\NRAG\\data\\01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'total_pages': 3,
  'text': 'Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter\'s 7-second skim.\n\n1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets rejected by a\nperson; a beautifully designed two-column resume may impress a human but fail to parse at all.\nWhat ac

In [6]:
import hashlib

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=180,
    separators=["\n\n", "\n", ". ", " "]
)

def make_chunk_id(file_name: str, page_number: int, text: str) -> str:
    # Content hash (not a running counter) -> re-running this notebook on
    # unchanged PDFs always produces the same IDs, so the indexing step
    # below can skip chunks that are already in the database instead of
    # re-embedding everything from scratch every time.
    digest = hashlib.sha256(f"{file_name}|{page_number}|{text}".encode("utf-8")).hexdigest()[:16]
    return f"{file_name}_p{page_number}_{digest}"

chunks = []

for record in cleaned_records:
    splits = text_splitter.split_text(record["text"])
    for s in splits:
        stripped = s.strip()
        if len(stripped) > 40:
            chunks.append({
                "chunk_id": make_chunk_id(record["file_name"], record["page_number"], stripped),
                "file_name": str(record["file_name"]),
                "page_number": int(record["page_number"]),
                "text": stripped
            })

print(f"✓ Generated {len(chunks)} cohesive chunks with context overlap.")

✓ Generated 73 cohesive chunks with context overlap.


In [7]:
chunks[:3]

[{'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_77b0c04da0e7f9e6',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'text': "Resume Writing Best Practices\n CareerForge Knowledge Base · Volume 1 · 2025-2026 Edition\nTarget reader: Anyone actively applying to roles who wants a resume that survives both AI/ATS screening\nand a human recruiter's 7-second skim."},
 {'chunk_id': '01_Resume_Writing_Best_Practices.pdf_p1_8eaf2df07ded7dce',
  'file_name': '01_Resume_Writing_Best_Practices.pdf',
  'page_number': 1,
  'text': '1. The Two Readers Every Resume Has\nEvery resume submitted online today is read twice: once by software (an Applicant Tracking System, often\nlayered with an AI scoring or summarization step), and once -- if it passes -- by a human recruiter who\ntypically spends under 10 seconds on a first pass. Optimizing for one at the expense of the other backfires.\nA resume stuffed with keywords may pass a naive parser but reads as robotic and gets

In [8]:
import chromadb
from chromadb.utils import embedding_functions

os.makedirs(CHROMA_PATH, exist_ok=True)

client = chromadb.PersistentClient(path=str(CHROMA_PATH))

# Lightweight local embedding model -- no API key, runs on CPU
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

collection = client.get_or_create_collection(
    name="career_knowledge_base",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Incremental indexing: only embed chunks whose ID isn't already stored.
# Re-running this cell after adding a new PDF costs almost nothing instead
# of re-embedding the whole knowledge base every time.
batch_size = 64
all_ids = [c["chunk_id"] for c in chunks]
existing_ids = set()
for i in range(0, len(all_ids), batch_size):
    batch_ids = all_ids[i:i + batch_size]
    existing_ids.update(collection.get(ids=batch_ids, include=[])["ids"])

new_chunks = [c for c in chunks if c["chunk_id"] not in existing_ids]

for i in range(0, len(new_chunks), batch_size):
    batch = new_chunks[i:i + batch_size]
    collection.add(
        ids=[c["chunk_id"] for c in batch],
        documents=[c["text"] for c in batch],
        metadatas=[{"file_name": c["file_name"], "page_number": c["page_number"]} for c in batch]
    )

print(f"✓ Indexed {len(new_chunks)} new chunk(s). Total chunks in DB: {collection.count()}")

✓ Indexed 73 new chunk(s). Total chunks in DB: 73


In [9]:
from rank_bm25 import BM25Okapi

# Common words carry no keyword-matching signal and would make BM25 see
# "shared vocabulary" between almost any query and any document (e.g. "the",
# "is", "of" appear everywhere) -- that would silently defeat the
# BM25_SCORE_THRESHOLD relevance gate used later.
_STOPWORDS = {
    "a", "an", "the", "and", "or", "but", "if", "then", "of", "to", "in", "on",
    "for", "with", "as", "is", "are", "was", "were", "be", "been", "being",
    "this", "that", "these", "those", "it", "its", "at", "by", "from", "into",
    "about", "what", "which", "who", "whom", "how", "when", "where", "why",
    "do", "does", "did", "you", "your", "i", "we", "they", "he", "she", "them",
    "his", "her", "their", "our", "my", "me", "us", "not", "no", "so", "such",
    "than", "too", "very", "can", "will", "would", "should", "could", "may",
    "might", "must", "have", "has", "had",
}

def tokenize(text: str) -> list[str]:
    return [w for w in re.findall(r"\w+", text.lower()) if w not in _STOPWORDS and len(w) > 1]

# Lexical (keyword) index, built fresh from `chunks` each run -- this is
# what makes hybrid search work: vector search finds semantically similar
# text even with no shared words, BM25 finds exact keyword/term matches
# even when the wording differs from typical phrasing. Combining both is
# more robust than either alone.
chunk_token_sets = [set(tokenize(c["text"])) for c in chunks]
bm25_index = BM25Okapi([tokenize(c["text"]) for c in chunks])

print(f"✓ Built BM25 lexical index over {len(chunks)} chunks.")

✓ Built BM25 lexical index over 73 chunks.


In [10]:
# A vector match below this is unreliable (an unrelated query still scores
# ~10-15% against any collection, since cosine similarity rarely hits 0).
# A BM25 score of 0 means literally no shared vocabulary with the query.
# A chunk is kept only if AT LEAST ONE search method is confident about it --
# that combination is what lets the system say "I don't know" instead of
# forcing an answer out of irrelevant context.
VECTOR_SIM_THRESHOLD = 30.0
BM25_SCORE_THRESHOLD = 1.0
RRF_K = 60  # standard constant for Reciprocal Rank Fusion, no tuning needed

def retrieve_relevant_chunks(query: str, n_results: int = 4, candidate_pool: int = 10) -> list[dict]:
    """Hybrid retrieval: fuse dense vector search with sparse BM25 keyword search.

    Each method ranks its own top `candidate_pool` chunks; a chunk's final
    score is the sum of 1/(RRF_K + rank) across whichever list(s) it
    appears in (Reciprocal Rank Fusion) -- a simple, scale-free way to
    combine two very differently-scaled ranking signals without manual
    weight tuning. Chunks that neither method is confident about are
    dropped before generation.
    """
    candidates: dict[str, dict] = {}

    vector_res = collection.query(
        query_texts=[query], n_results=candidate_pool,
        include=["documents", "metadatas", "distances"]
    )
    for rank, (cid, doc, meta, dist) in enumerate(zip(
        vector_res["ids"][0], vector_res["documents"][0],
        vector_res["metadatas"][0], vector_res["distances"][0]
    )):
        candidates[cid] = {
            "chunk_id": cid, "text": doc,
            "file_name": meta["file_name"], "page_number": meta["page_number"],
            "vector_sim": round((1 - dist) * 100, 2), "vector_rank": rank,
            "bm25_score": 0.0, "bm25_rank": None,
        }

    # On a small corpus, a single generic word (e.g. "point") can be rare
    # enough to get a high BM25/IDF score by accident, even though it's not
    # a real topical match. Requiring at least 2 shared query terms (or all
    # of them, for a 1-term query) filters that out without needing an
    # ever-growing stopword list.
    query_tokens = tokenize(query)
    query_token_set = set(query_tokens)
    min_overlap = min(2, len(query_token_set)) if query_token_set else 0

    bm25_scores = bm25_index.get_scores(query_tokens)
    bm25_top = sorted(range(len(chunks)), key=lambda i: bm25_scores[i], reverse=True)[:candidate_pool]
    for rank, idx in enumerate(bm25_top):
        if len(query_token_set & chunk_token_sets[idx]) < min_overlap:
            continue
        c = chunks[idx]
        entry = candidates.setdefault(c["chunk_id"], {
            "chunk_id": c["chunk_id"], "text": c["text"],
            "file_name": c["file_name"], "page_number": c["page_number"],
            "vector_sim": 0.0, "vector_rank": None,
            "bm25_score": 0.0, "bm25_rank": None,
        })
        entry["bm25_score"] = round(float(bm25_scores[idx]), 2)
        entry["bm25_rank"] = rank

    def rrf(rank):
        return 0.0 if rank is None else 1.0 / (RRF_K + rank + 1)

    relevant = []
    for c in candidates.values():
        c["score"] = rrf(c["vector_rank"]) + rrf(c["bm25_rank"])
        if c["vector_sim"] >= VECTOR_SIM_THRESHOLD or c["bm25_score"] >= BM25_SCORE_THRESHOLD:
            relevant.append(c)

    relevant.sort(key=lambda c: c["score"], reverse=True)
    return relevant[:n_results]

def search_and_display(query: str, n_results: int = 4):
    """Pretty-print hybrid search results for quick manual inspection."""
    print(f"\nSearching for: '{query}'")
    print("=" * 70)

    hits = retrieve_relevant_chunks(query, n_results=n_results)
    if not hits:
        print("No sufficiently relevant chunks found (vector + BM25 both below threshold).")
        return

    for idx, hit in enumerate(hits, start=1):
        print(
            f"Result #{idx} | vector={hit['vector_sim']}% bm25={hit['bm25_score']} "
            f"| Source: {hit['file_name']} (Page {hit['page_number']})"
        )
        print("-" * 70)
        print(hit["text"])
        print("=" * 70)

test_query = "How should I structure my resume bullet points to show measurable impact?"
search_and_display(test_query, n_results=4)


Searching for: 'How should I structure my resume bullet points to show measurable impact?'
Result #1 | vector=53.14% bm25=7.9 | Source: 02_How_to_Analyze_a_Job_Description.pdf (Page 2)
----------------------------------------------------------------------
5. Turning the Analysis Into Action

Rewrite your resume's Core Competencies section using the JD's own terminology for skills you
genuinely have.

Reorder your bullet points within each role so the most JD-relevant achievements appear first.

Draft 2-3 STAR stories (see the Behavioral Interview guide) that map directly to the top 3 requirements
in the posting.

If applying via a portal, paste your tailored resume and the JD into a keyword-comparison tool if one is
available to catch obvious gaps before submitting.
Result #2 | vector=53.86% bm25=4.92 | Source: 01_Resume_Writing_Best_Practices.pdf (Page 2)
----------------------------------------------------------------------
Projects / Portfolio (optional but increasingly expected in

In [11]:
import ollama

OLLAMA_MODEL = "llama3.2:3b"
OLLAMA_TIMEOUT_SECONDS = 120  # local CPU generation can be slow; still bounded, never infinite
MAX_HISTORY_TURNS = 3  # how many past Q&A pairs to keep as conversational context

# A plain Client with a timeout -- the bare `ollama.chat(...)` shortcut used
# by the previous version of this notebook has no timeout, so a stuck/
# overloaded server would hang the cell forever with no way out.
_ollama_client = ollama.Client(timeout=OLLAMA_TIMEOUT_SECONDS)

# Q&A pairs only (not the large retrieved-context blob) -- keeps memory
# small while still letting the model handle a natural follow-up question.
conversation_history: list[dict] = []

def reset_conversation():
    conversation_history.clear()


def build_context(retrieved_chunks: list[dict]) -> str:
    """Format retrieved chunks with source metadata for the prompt.

    Deliberately does NOT label chunks as "Document [1]", "Document [2]",
    etc. -- earlier versions of this prompt did, and the model would then
    cite "(Document [2], Page 2)" in its answer instead of the real file
    name, defeating the whole point of the citation rule below.
    """
    context_blocks = []
    for chunk in retrieved_chunks:
        block = (
            f"--- Source: {chunk['file_name']} (Page {chunk['page_number']}) ---\n"
            f"{chunk['text'].strip()}"
        )
        context_blocks.append(block)

    return "\n\n".join(context_blocks)


SYSTEM_INSTRUCTION = (
    "You are an expert career consultant. Answer the user's question directly and "
    "comprehensively using ONLY the provided context, and take prior turns in this "
    "conversation into account when relevant (e.g. follow-up questions).\n"
    "Strict Rules:\n"
    "1. NEVER invent, extrapolate, or fabricate any examples. If an example is provided "
    "in the text, quote or adapt ONLY that exact example.\n"
    "2. Present any formula or framework given in the text (e.g. the X-Y-Z bullet formula) "
    "completely, along with its accompanying rules.\n"
    "3. Every paragraph or piece of advice MUST end with an explicit source citation in "
    "EXACTLY this format: (exact_file_name.pdf, Page X) -- using the real file name shown "
    "in the 'Source:' header above each context block. For example: "
    "(05_Salary_Negotiation_Playbook.pdf, Page 1). Never write a placeholder like 'Document [1]', "
    "and never add extra words inside the parentheses.\n"
    "4. If the context does not contain enough information to answer, say so explicitly "
    "instead of guessing."
)

_PAREN_PATTERN = re.compile(r"\(([^()]+)\)")
_FILE_PATTERN = re.compile(r"([\w\-]+\.pdf)", re.IGNORECASE)
_PAGE_PATTERN = re.compile(r"Page\s*(\d+)", re.IGNORECASE)


def find_unverifiable_citations(answer: str, retrieved_chunks: list[dict]) -> list[str]:
    """Flag citations in the answer that don't match any retrieved (file, page).

    Looks for a *.pdf filename and a "Page N" inside each parenthetical
    independently (rather than requiring one exact format), so a harmless
    variation like "(Source: file.pdf, Page 3)" isn't a false positive --
    only a wrong/hallucinated file name or page number is flagged.

    Note: this only verifies that the citation *points* somewhere real --
    it does not check that the sentence next to it is actually supported
    by that chunk's content. Full claim-level grounding needs a much
    heavier verifier (e.g. a second LLM call or an NLI model) and is left
    out here to keep the notebook simple.
    """
    valid_pairs = {(c["file_name"].lower(), c["page_number"]) for c in retrieved_chunks}
    problems = []
    for paren_match in _PAREN_PATTERN.finditer(answer):
        content = paren_match.group(1)
        file_match = _FILE_PATTERN.search(content)
        page_match = _PAGE_PATTERN.search(content)
        if not (file_match and page_match):
            continue  # not a citation-shaped parenthetical, e.g. "(Y)" in the X-Y-Z formula
        file_name, page = file_match.group(1).lower(), int(page_match.group(1))
        if (file_name, page) not in valid_pairs:
            problems.append(paren_match.group(0))
    if re.search(r"\bDocument\s*\[\d+\]", answer, re.IGNORECASE):
        problems.append("placeholder citation (e.g. 'Document [1]') instead of a real file name")
    return problems


def generate_rag_response(query: str, n_results: int = 5, use_history: bool = True) -> dict:
    retrieved_chunks = retrieve_relevant_chunks(query, n_results=n_results)

    if not retrieved_chunks:
        return {
            "answer": "No relevant information was found in the knowledge base for this question.",
            "sources": [],
            "citation_warnings": [],
        }

    formatted_context = build_context(retrieved_chunks)
    user_message = f"""Context Documents:
{formatted_context}

Question: {query}

Provide a structured, helpful answer based strictly on the context above:"""

    print(f"Generating answer using local {OLLAMA_MODEL}...")

    messages = [{"role": "system", "content": SYSTEM_INSTRUCTION}]
    if use_history:
        messages.extend(conversation_history[-2 * MAX_HISTORY_TURNS:])
    messages.append({"role": "user", "content": user_message})

    try:
        response = _ollama_client.chat(
            model=OLLAMA_MODEL,
            messages=messages,
            options={"temperature": 0.1}
        )
    except Exception as exc:
        return {
            "answer": (
                f"Could not reach Ollama model '{OLLAMA_MODEL}'. "
                f"Make sure `ollama serve` is running and the model is pulled "
                f"(`ollama pull {OLLAMA_MODEL}`). Details: {exc}"
            ),
            "sources": retrieved_chunks,
            "citation_warnings": [],
        }

    answer = response["message"]["content"]
    citation_warnings = find_unverifiable_citations(answer, retrieved_chunks)

    if use_history:
        conversation_history.append({"role": "user", "content": query})
        conversation_history.append({"role": "assistant", "content": answer})

    return {"answer": answer, "sources": retrieved_chunks, "citation_warnings": citation_warnings}

In [12]:
# ====================================================
# End-to-end test
# ====================================================
test_query = "How should I structure my resume bullet points to show measurable impact?"

result = generate_rag_response(test_query, n_results=4)

print("\n" + "=" * 70)
print("🤖 Final Local RAG Response:")
print("=" * 70)
print(result["answer"])

if result["citation_warnings"]:
    print("\n⚠️  Unverifiable citation(s) detected:")
    for warning in result["citation_warnings"]:
        print(f"   - {warning}")

Generating answer using local llama3.2:3b...

🤖 Final Local RAG Response:
According to the provided context, specifically from the section "Writing Bullets That Actually Land" in the document 01_Resume_Writing_Best_Practices.pdf (Page 3), you should use the X-Y-Z formula to structure your resume bullet points and show measurable impact. The formula is as follows:

X-Y-Z formula: Accomplished [X], measured by [Y], by doing [Z].

This means that you should:

1. Start with a strong action verb (e.g., "Reduced", "Built", "Automated", etc.)
2. Describe the accomplishment (X)
3. Specify the metric or outcome that measures the impact (Y)
4. Explain the specific action or effort taken to achieve the outcome (Z)

For example: "Reduced customer churn by 18% (Y) by redesigning the onboarding email sequence (Z), resulting in $240K in retained annual revenue (X)."


In [13]:
# ====================================================
# Demo: conversation memory + the "I don't know" relevance gate
# ====================================================
def ask(question: str):
    result = generate_rag_response(question, n_results=4)
    print("\n" + "=" * 70)
    print(f"🤖 Q: {question}")
    print("=" * 70)
    print(result["answer"])
    if result["sources"]:
        print("\nSources:", ", ".join(f"{s['file_name']} p{s['page_number']}" for s in result["sources"]))
    if result["citation_warnings"]:
        print("⚠️  Unverifiable citation(s):", result["citation_warnings"])

reset_conversation()

# Turn 1
ask("What's a good salary negotiation tactic?")

# Turn 2 -- a follow-up that only makes sense with memory of turn 1
ask("Can you give me one more tip like that?")

# Turn 3 -- off-topic: the hybrid relevance gate should now refuse instead
# of fabricating an answer from irrelevant chunks (this used to silently
# "succeed" with 9-14% similarity matches before the threshold was added).
ask("What is the boiling point of water on Mars?")

Generating answer using local llama3.2:3b...

🤖 Q: What's a good salary negotiation tactic?
According to the 05_Salary_Negotiation_Playbook.pdf (Page 1), a good salary negotiation tactic is to "Let them anchor first when possible." This means that the negotiator should try to get the other party to state a number first, as this can give away information and limit the negotiator's room for maneuver. By letting the other party anchor, the negotiator can gain an advantage and potentially negotiate a better deal.

Sources: 05_Salary_Negotiation_Playbook.pdf p1, 05_Salary_Negotiation_Playbook.pdf p2, 05_Salary_Negotiation_Playbook.pdf p1, 05_Salary_Negotiation_Playbook.pdf p1
Generating answer using local llama3.2:3b...

🤖 Q: Can you give me one more tip like that?
Another tip is to "Mirror the company's language" in your cover letter or summary, as mentioned in 02_How_to_Analyze_a_Job_Description.pdf (Page 1). This means using the same words and phrases that the company uses in their job d